# Linear(선형) 양자화 실습

파이토치로 MNIST를 MLP로 간단히 학습시킨 뒤, 학습된 가중치를 **선형(아핀) 양자화**해 봅니다.

1. 가중치를 safetensors 파일로 저장/로드
2. 가중치 분포 히스토그램
3. **Linear 양자화** (torch.fake_quantize) — 아핀 매핑 r = S(q − Z), 균등 격자
4. 비트 수·모드(asymmetric/symmetric)에 따른 양자화 오차·정확도

Colab에서는 위에서부터 순서대로 실행하면 됩니다. (런타임 → 모두 실행)

## 0. 준비

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

try:
    from safetensors.torch import save_file, load_file
except ImportError:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "safetensors"])
    from safetensors.torch import save_file, load_file

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. MNIST를 MLP로 학습

작은 3층 MLP(784 → 256 → 128 → 10)를 2 에폭만 학습합니다. 목적은 최고 성능이 아니라 **양자화해 볼 가중치**를 얻는 것이므로 짧게 돌립니다.

In [ ]:
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.1307,), (0.3081,))])
train_ds = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=transform)
test_ds  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

model = MLP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

def accuracy(m):
    m.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (m(x).argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total

for epoch in range(2):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        F.cross_entropy(model(x), y).backward()
        opt.step()
    print(f"epoch {epoch+1}  test acc = {accuracy(model):.4f}")

base_acc = accuracy(model)
print("baseline (float32) accuracy:", base_acc)

# 학습된 가중치를 safetensors 파일로 저장
save_file({k: v.cpu().contiguous() for k, v in model.state_dict().items()},
          "mnist_mlp.safetensors")
print("saved -> mnist_mlp.safetensors")

## 2. 가중치 파일 저장하고 다시 불러오기 (safetensors)

임의로 만든 행렬이 아니라, 방금 **파일로 저장한 실제 체크포인트**(`mnist_mlp.safetensors`)를 다시 읽어 다뤄봅니다.

`.pth`(pickle 기반) 대신 **safetensors** 포맷을 씁니다.

- **안전**: pickle이 아니라 임의 코드 실행 위험이 없음
- **빠름**: zero-copy 로딩, 필요한 텐서만 골라 읽기 가능
- **표준**: HuggingFace 모델 가중치의 사실상 표준 포맷

> 이미 가진 `.safetensors` 파일이 있다면 Colab 파일 탭에 업로드한 뒤 경로만 바꿔서 그대로 실습할 수 있습니다.

In [ ]:
import os
print("file size:", os.path.getsize("mnist_mlp.safetensors"), "bytes")

# safetensors 파일에서 state_dict 로드 (텐서는 CPU로 반환됨)
state = load_file("mnist_mlp.safetensors")
print("\nstate_dict keys / shapes:")
for k, v in state.items():
    print(f"  {k:14s} {tuple(v.shape)}  ({v.numel()} params)")

# 새 모델에 로드해서 그대로 사용
model = MLP().to(device)
model.load_state_dict(load_file("mnist_mlp.safetensors"))
model.eval()
print("\nreloaded model test acc:", accuracy(model))

## 3. 가중치 분포 히스토그램

파일에서 불러온 첫 번째 층(`fc1`)의 가중치를 펼쳐 히스토그램을 그려봅니다. 신경망 가중치의 전형적인 특징 — **0 근처에 종 모양으로 몰려 있고 꼬리가 긴** 분포 — 를 확인할 수 있습니다.

In [ ]:
w = model.fc1.weight.detach().cpu().numpy()
print("fc1.weight shape:", w.shape, " (총", w.size, "개 파라미터)")
print(f"min={w.min():.3f}  max={w.max():.3f}  mean={w.mean():.4f}  std={w.std():.3f}")

plt.figure(figsize=(7, 3.5))
plt.hist(w.reshape(-1), bins=200, color="#3b82f6")
plt.title("fc1 weight distribution (float32)")
plt.xlabel("weight value"); plt.ylabel("count")
plt.tight_layout(); plt.show()

## 4. Linear 양자화 (아핀 매핑)

정수 $q$와 실수 $r$을 아핀 매핑으로 잇습니다.

$$r = S \cdot (q - Z), \qquad S = \frac{r_{\max}-r_{\min}}{q_{\max}-q_{\min}}, \qquad Z = \mathrm{round}\!\left(q_{\min} - \frac{r_{\min}}{S}\right)$$

- **균등(uniform)**: 대표값들이 일정한 간격 $S$로 배치됨
- `symmetric=True`면 $Z=0$ 고정 (가중치가 0 대칭일 때 유리, 정수 연산이 더 단순)

`S`, `Z`는 min/max로 계산하고, 양자화(round·clamp)와 복원(dequant)은 **PyTorch 내장 연산** `torch.fake_quantize_per_tensor_affine`이 처리합니다. (임의 비트 폭을 `quant_min`/`quant_max`로 지정 가능)

In [ ]:
def linear_quantize(w, n_bits, symmetric=False):
    """PyTorch 내장 fake_quantize로 선형(아핀) 양자화. 복원값과 S, Z를 반환."""
    t = torch.as_tensor(w, dtype=torch.float32)
    if symmetric:
        qmax = 2 ** (n_bits - 1) - 1
        qmin = -qmax
        S = float(t.abs().max()) / qmax
        Z = 0
    else:
        qmin = -2 ** (n_bits - 1)
        qmax = 2 ** (n_bits - 1) - 1
        S = (float(t.max()) - float(t.min())) / (qmax - qmin)
        Z = int(round(qmin - float(t.min()) / S))
    recon = torch.fake_quantize_per_tensor_affine(t, S, Z, qmin, qmax)
    return recon.numpy(), S, Z

N_BITS = 2
recon_lin, S, Z = linear_quantize(w, N_BITS, symmetric=False)
mse_lin = ((w - recon_lin) ** 2).mean()
print(f"[Linear] {N_BITS}-bit (asymmetric)")
print(f"scale S = {S:.5f}   zero point Z = {Z}")
print("levels:", np.unique(recon_lin.reshape(-1)).round(3))
print(f"quantization MSE = {mse_lin:.6f}")

plt.figure(figsize=(7, 3.5))
plt.hist(w.reshape(-1), bins=200, alpha=0.4, color="#3b82f6", label="original")
plt.hist(recon_lin.reshape(-1), bins=200, color="#22c55e", label="Linear quantized")
for lv in np.unique(recon_lin.reshape(-1)):
    plt.axvline(lv, color="#22c55e", lw=1, alpha=0.7)
plt.title(f"Linear quantization ({N_BITS}-bit, uniform levels)")
plt.xlabel("weight value"); plt.ylabel("count"); plt.legend()
plt.tight_layout(); plt.show()

## 5. 비트 수·모드에 따른 정확도

모델의 모든 Linear 층 가중치를 선형 양자화한 뒤 MNIST 테스트 정확도를 비트 수와 모드(asymmetric / symmetric)별로 비교합니다.

In [ ]:
import copy

def eval_linear(model, n_bits, symmetric):
    m = copy.deepcopy(model)
    with torch.no_grad():
        for layer in [m.fc1, m.fc2, m.fc3]:
            wl = layer.weight.detach().cpu().numpy()
            recon, *_ = linear_quantize(wl, n_bits, symmetric=symmetric)
            layer.weight.copy_(torch.tensor(recon, dtype=torch.float32, device=device))
    return accuracy(m)

bits_list = [2, 3, 4, 8]
acc_asym = [eval_linear(model, b, False) for b in bits_list]
acc_sym  = [eval_linear(model, b, True)  for b in bits_list]

print(f"{'bits':>5} | {'asymmetric':>11} | {'symmetric':>10}")
for b, a1, a2 in zip(bits_list, acc_asym, acc_sym):
    print(f"{b:>5} | {a1:>11.4f} | {a2:>10.4f}")
print(f"{'float':>5} | {base_acc:>11.4f} | {base_acc:>10.4f}")

plt.figure(figsize=(6.5, 4))
plt.axhline(base_acc, color="gray", ls=":", label=f"float32 ({base_acc:.3f})")
plt.plot(bits_list, acc_asym, "o-", color="#22c55e", label="asymmetric")
plt.plot(bits_list, acc_sym,  "s--", color="#0ea5a5", label="symmetric (Z=0)")
plt.title("MNIST test accuracy vs bit-width (Linear)")
plt.xlabel("bits per weight"); plt.ylabel("test accuracy")
plt.xticks(bits_list); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 마무리 & 더 해보기

- Linear 양자화는 대표값을 **균등 간격**으로 놓습니다. 비균등(K-Means)보다 낮은 비트에서 오차가 클 수 있지만, **정수 연산으로 추론**까지 가능하다는 큰 장점이 있습니다.
- asymmetric은 $Z$로 범위를 정확히 맞추고, symmetric($Z=0$)은 정수 연산이 더 단순합니다.

**더 해볼 것**
1. 활성값(activation)까지 양자화해 정수 추론 파이프라인 구현
2. `torch.ao.quantization`으로 QAT(양자화 인지 학습) 적용
3. per-channel 양자화로 정확도 개선